In [31]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

In [32]:
df = pd.read_csv("Cleaned_Dataset/master_airline_loyalty.csv")

df.head()

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Country,Province,...,Gender,Education,Salary,Salary_College,Marital Status,Loyalty Card,CLV,Enrollment Year,Enrollment Month,Cancelled
0,100018,2017,1,1,601,601.0,0,0,Canada,Alberta,...,Female,Bachelor,92552.0,0,Married,Aurora,7919.2,2016,8,0
1,100018,2017,2,0,0,0.0,0,0,Canada,Alberta,...,Female,Bachelor,92552.0,0,Married,Aurora,7919.2,2016,8,0
2,100018,2017,3,4,9648,9648.0,438,79,Canada,Alberta,...,Female,Bachelor,92552.0,0,Married,Aurora,7919.2,2016,8,0
3,100018,2017,4,1,1654,1654.0,0,0,Canada,Alberta,...,Female,Bachelor,92552.0,0,Married,Aurora,7919.2,2016,8,0
4,100018,2017,5,0,0,0.0,0,0,Canada,Alberta,...,Female,Bachelor,92552.0,0,Married,Aurora,7919.2,2016,8,0


In [33]:
df.shape
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 389065 entries, 0 to 389064
Data columns (total 21 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Loyalty Number               389065 non-null  int64  
 1   Year                         389065 non-null  int64  
 2   Month                        389065 non-null  int64  
 3   Total Flights                389065 non-null  int64  
 4   Distance                     389065 non-null  int64  
 5   Points Accumulated           389065 non-null  float64
 6   Points Redeemed              389065 non-null  int64  
 7   Dollar Cost Points Redeemed  389065 non-null  int64  
 8   Country                      389065 non-null  object 
 9   Province                     389065 non-null  object 
 10  City                         389065 non-null  object 
 11  Gender                       389065 non-null  object 
 12  Education                    389065 non-null  object 
 13 

,Loyalty Number,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Salary,Salary_College,CLV,Enrollment Year,Enrollment Month,Cancelled
count,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000,389065.000000
mean,550237.484605,2017.513726,6.513726,1.307771,1960.756550,2047.341685,31.615725,5.691733,59384.455124,0.253474,7987.027279,2015.164093,6.786455,0.123663
std,258565.821972,0.499812,3.445396,1.977017,3262.194313,3897.263156,127.287188,22.915197,45447.667740,0.435001,6838.585184,1.947048,3.386384,0.329197
min,100018.000000,2017.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1898.010000,2012.000000,1.000000,0.000000
25%,327470.000000,2017.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3980.420000,2013.000000,4.000000,0.000000
50%,551510.000000,2018.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,63887.000000,0.000000,5773.520000,2015.000000,7.000000,0.000000
75%,772019.000000,2018.000000,10.000000,2.000000,3056.000000,3078.000000,0.000000,0.000000,83123.000000,1.000000,8954.430000,2017.000000,10.000000,0.000000
max,999986.000000,2018.000000,12.000000,32.000000,67284.000000,100926.000000,996.000000,179.000000,407228.000000,1.000000,83325.380000,2018.000000,12.000000,1.000000


In [34]:
df.drop(
    columns=["Loyalty Number"],
    inplace=True
)

## **Creating Important Features**

In [35]:
# Customer Tenure

df["Tenure"] = (
    df["Year"] -
    df["Enrollment Year"]
)

df["Tenure"] = df["Tenure"].clip(lower=0)

In [36]:
# Membership Age (Months)

df["Membership_Age_Months"] = (
        (df["Year"] * 12)
        + df["Month"]
) - (
        (df["Enrollment Year"] * 12)
        + df["Enrollment Month"]
)

df["Membership_Age_Months"] = (
    df["Membership_Age_Months"]
    .clip(lower=0)
)

In [37]:
# Redemption Ratio

df["Redemption_Ratio"] = (
        df["Points Redeemed"]
        /
        (df["Points Accumulated"] + 1)
)

In [38]:
# Average Distance Per Flight

df["Avg_Distance_Per_Flight"] = (
        df["Distance"]
        /
        (df["Total Flights"] + 1)
)

In [39]:
# Points Per Flight

df["Points_Per_Flight"] = (
        df["Points Accumulated"]
        /
        (df["Total Flights"] + 1)
)

In [40]:
# Redeemed Cost Per Point

df["Redeemed_Cost_Per_Point"] = (
        df["Dollar Cost Points Redeemed"]
        /
        (df["Points Redeemed"] + 1)
)

In [41]:
# Premium Customer Flag

premium_cards = [
    "Aurora",
    "Nova"
]

df["Premium_Customer"] = np.where(
    df["Loyalty Card"].isin(premium_cards),
    1,
    0
)

In [42]:
df["Flight_Segment"] = pd.qcut(
    df["Total Flights"],
    q=3,
    labels=["Low", "Medium"],
    duplicates="drop"
)

In [43]:
# Flight Frequency Segment

def flight_segment(flights):
    if flights <= 2:
        return "Low"
    elif flights <= 6:
        return "Medium"
    else:
        return "High"

df["Flight_Segment"] = df["Total Flights"].apply(flight_segment)

In [44]:
# CLV Segment

df["CLV_Segment"] = pd.qcut(
    df["CLV"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "VIP"
    ]
)

In [45]:
# Engagement Score

from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler()

cols = [
    "Total Flights",
    "Distance",
    "Redemption_Ratio"
]

df[cols] = mms.fit_transform(
    df[cols]
)

In [46]:
df["Engagement_Score"] = (
        0.4 * df["Total Flights"]
        +
        0.3 * df["Distance"]
        +
        0.3 * df["Redemption_Ratio"]
)

In [47]:
# Travel Intensity

df["Travel_Intensity"] = (
        df["Distance"]
        /
        (df["Tenure"] + 1)
)

In [48]:
# Reward Dependency

df["Reward_Dependency"] = (
        df["Points Redeemed"]
        /
        (df["CLV"] + 1)
)

In [49]:
# Loyalty Efficiency

df["Loyalty_Efficiency"] = (
        df["CLV"]
        /
        (df["Total Flights"] + 1)
)

## **Frequency Encoding**

In [50]:
# Frequency Encoding
# City
city_freq = (
    df["City"]
    .value_counts(normalize=True)
)

df["City_FE"] = (
    df["City"]
    .map(city_freq)
)

In [51]:
# Province

province_freq = (
    df["Province"]
    .value_counts(normalize=True)
)

df["Province_FE"] = (
    df["Province"]
    .map(province_freq)
)

In [52]:
# Country

country_freq = (
    df["Country"]
    .value_counts(normalize=True)
)

df["Country_FE"] = (
    df["Country"]
    .map(country_freq)
)

## **One-Hot Encoding**

In [53]:
low_card_cols = [
    "Gender",
    "Education",
    "Marital Status",
    "Loyalty Card",
    "Flight_Segment",
    "CLV_Segment"
]

df = pd.get_dummies(
    df,
    columns=low_card_cols,
    drop_first=True
)

In [54]:
df.to_csv(
    "airline_feature_engineered.csv",
    index=False
)

print("Feature Engineered Dataset Saved Successfully!")

Feature Engineered Dataset Saved Successfully!


In [55]:
# Save Feature Engineered Dataset

df.to_csv(
    "airline_feature_engineered.csv",
    index=False
)

In [56]:
df = pd.read_csv("airline_feature_engineered.csv")

print(df.shape)
df.head()

(389065, 44)


,Year,Month,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,Country,Province,City,...,Education_Master,Marital Status_Married,Marital Status_Single,Loyalty Card_Nova,Loyalty Card_Star,Flight_Segment_Low,Flight_Segment_Medium,CLV_Segment_Medium,CLV_Segment_High,CLV_Segment_VIP
0,2017,1,0.03125,0.008932,601.0,0,0,Canada,Alberta,Edmonton,...,False,True,False,False,False,True,False,False,True,False
1,2017,2,0.00000,0.000000,0.0,0,0,Canada,Alberta,Edmonton,...,False,True,False,False,False,True,False,False,True,False
2,2017,3,0.12500,0.143392,9648.0,438,79,Canada,Alberta,Edmonton,...,False,True,False,False,False,False,True,False,True,False
3,2017,4,0.03125,0.024582,1654.0,0,0,Canada,Alberta,Edmonton,...,False,True,False,False,False,True,False,False,True,False
4,2017,5,0.00000,0.000000,0.0,0,0,Canada,Alberta,Edmonton,...,False,True,False,False,False,True,False,False,True,False
